# Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
In this notebook, we demonstrate how vLLM-Hook enables **Activation Steering** for controlled generation.

**Paper**: [Improving Instruction-Following in Language Models through Activation Steering](https://arxiv.org/abs/2410.12877).<br />
**Authors**: Alessandro Stolfo, Vidhisha Balachandran, Safoora Yousefi, Eric Horvitz, Besmira Nushi <br />
**"TL;DR"**: Activation steering allows you to bias the model's behavior by nudging internal activations in specific directions. In this paper, authors focus on instruction following capability and compute the steering vectors as the difference in activations between inputs with and without instructions. 

### Installation

The cell below installs MIA. Run from a clone of the repository (this notebook sits in `notebooks/`), it installs the validated dependencies from `requirement.txt` and the checkout in editable mode; anywhere else it installs `vllm-hook` from PyPI.<br />
It is not necessary if the package is already installed in this environment.

In [ ]:
from pathlib import Path

# In a clone, this notebook is in <repo>/notebooks/ next to requirement.txt.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REQ_FILE = REPO_ROOT / "requirement.txt"

if REQ_FILE.exists() and (REPO_ROOT / "pyproject.toml").exists():
    print("Repo checkout found at", REPO_ROOT)
    %pip install -r "{REQ_FILE}"
    %pip uninstall -y torchcodec
    %pip install --no-deps -e "{REPO_ROOT}"
else:
    print("No repo checkout found: installing vllm-hook from PyPI")
    %pip install vllm-hook

### Importing the MIA LLM
MIA provides `MiaLLM`, a wrapper around `vllm.LLM` that arms a capture or steering worker and runs an analyzer on what it captures.
We import it here together with `config_path`, which finds the example configs and steering vectors whether MIA came from a clone or from PyPI:

In [ ]:
from mia import MiaLLM, config_path

### Environment & multiprocessing setup

In [ ]:
import os
import multiprocessing as mp
import torch
from pathlib import Path
from vllm import SamplingParams
mp.set_start_method("spawn", force=True)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


### Initialize `MiaLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [ ]:
cache_dir = os.path.expanduser('~/.cache/mia')  # Specify cache dir
model = 'microsoft/Phi-3-mini-4k-instruct'

dtype_map = {
    'microsoft/Phi-3-mini-4k-instruct': 'auto',
}

We also need to provide a config file that specifies how activations are steered (e.g., which layers to intervene on, which token to intervene, what direction vectors to apply, etc.).<br />
In the following example, we apply activation steering at the 15th layer, apply the steering at all positions (as opposed to only at the start of the decoding process), and along the direction given in `vector_path`:

In [ ]:
import json

json_path = Path(config_path("activation_steer/Phi-3-mini-4k-instruct.json"))  # adjust path

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)

Inside the `steer` worker, MIA defines the activation steering behavior during model inference.
Now, we initialize the llm:

In [ ]:
llm = MiaLLM(
    model=model,
    worker_name="steer",
    config_file=str(json_path),
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    max_model_len=4096,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_prefix_caching=True,
    enable_hook=True,
)

### Test case
In the following, we show a test case and compare generations **with** and **without** activation steering.

**Note**: Users should swap the example configs with their own to show desirable performance. The following is for pipeline illustration only.

In [6]:
test_cases = [
    "Write a dialogue between two people, one is dressed up in a ball gown and the other is dressed down in sweats. The two are going to a nightly event. Your answer must contain exactly 3 bullet points in the markdown format (use \"* \" to indicate each bullet) such as:\n* This is the first point.\n* This is the second point.",
    "What is the difference between the 13 colonies and the other British colonies in North America? Your answer must contain exactly 6 bullet point in Markdown using the following format:\n* Bullet point one.\n* Bullet point two.\n...\n* Bullet point fix."
]

Before we start, we define the sampling parameters:

**Note**: token 32007 is phi-specific, refer to the original huggingface implementation for details https://github.com/microsoft/llm-steer-instruct/blob/main/utils/generation_utils.py.

- Option 1: we can use a uniform sampling parameters as follows

In [7]:
# sampling_params = SamplingParams(
#     temperature=0.0,                       
#     max_tokens=2048,
#     stop_token_ids=[llm.tokenizer.eos_token_id, 32007],  
# )

- Option 2: we can vary the steering settings per request by passing a list of `SamplingParams` aligned with the prompts. Each entry's `extra_args["steer"]` carries the full steering config for that request, so different prompts in the **same batch** can use different coefficients, methods, optimal layers, etc. Below, the first prompt uses the JSON config as-is, while the second prompt overrides `method` and `coefficient` (everything else inherited from the JSON config).

In [8]:
base_steer = config["steering"]  # full default config from the JSON

sampling_params_list = [
    SamplingParams(
        temperature=0.0,
        max_tokens=2048,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
    ),
    SamplingParams(
        temperature=0.0,
        max_tokens=2048,
        stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
        extra_args={"steer": {**base_steer, "method": "add_vector", "coefficient": 10}},
    ),
]

Next we:
1. Apply chat template on each test case,
2. Generate all prompts in one batched call with their per-request `SamplingParams`. With activation steering enabled (`use_hook=True`, default), each prompt's hook reads its own `extra_args["steer"]` so different prompts in the same batch use different steer settings,
3. Reset the prefix cache to ensure the baseline generation does not reuse steered cache,
4. Generate again with `use_hook=False` to obtain the baseline output.

In [9]:
examples = [
    llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": case}], add_generation_prompt=True, tokenize=False
    )
    for case in test_cases
]

outputs = llm.generate(examples, sampling_params_list)

llm.llm_engine.reset_prefix_cache()

outputs_original = llm.generate(examples, sampling_params_list, use_hook=False)

Processed prompts: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.41s/it, est. speed input: 51.73 toks/s, output: 128.26 toks/s]


(EngineCore pid=3981266) Installed 32 steering hooks on layers: ['model.layers.0', 'model.layers.1', 'model.layers.2', 'model.layers.3', 'model.layers.4', 'model.layers.5', 'model.layers.6', 'model.layers.7', 'model.layers.8', 'model.layers.9', 'model.layers.10', 'model.layers.11', 'model.layers.12', 'model.layers.13', 'model.layers.14', 'model.layers.15', 'model.layers.16', 'model.layers.17', 'model.layers.18', 'model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28', 'model.layers.29', 'model.layers.30', 'model.layers.31']
(EngineCore pid=3981266) Hooks installed successfully
(EngineCore pid=3981266) INFO 06-01 23:12:36 [block_pool.py:472] Successfully reset prefix cache


Processed prompts: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.04s/it, est. speed input: 70.09 toks/s, output: 156.98 toks/s]


Finally we can print out the results as follows:

In [10]:
for steered, original in zip(outputs, outputs_original):
    print("=" * 100)
    steered_text = steered.outputs[0].text
    print("\n[With activation steering]\n")
    print(steered_text)
    
    baseline_text = original.outputs[0].text
    print("\n[Without activation steering]\n")
    print(baseline_text)


[With activation steering]

 * The woman in the ball gown is wearing a stunning, floor-length dress with intricate lace details and matching high heels.
* The man in the sweats is wearing a comfortable, casual t-shirt and sweatpants, with sneakers on his feet.
* They both realize they are going to a nightly event and start discussing what they should wear to make a good impression.

[Without activation steering]

 * The woman in the ball gown is excitedly discussing the upcoming event with her friend, who is dressed in casual sweats.
* The friend in sweats expresses their discomfort with the formal attire, but the woman in the ball gown reassures them that they will have a great time regardless of what they wear.
* The two friends make plans to meet at the event, with the woman in the ball gown promising to help her friend find a comfortable and stylish outfit for the occasion.

[With activation steering]

 * The 13 colonies, also known as the Thirteen Colonies, were British colonies 